In [1]:
from Declare4Py.ProcessModels.DeclareModel import DeclareModel
from Declare4Py.ProcessMiningTasks.Discovery.DeclareMiner import DeclareMiner
from Declare4Py.D4PyEventLog import D4PyEventLog
from Declare4Py.ProcessModels.DeclareModel import DeclareModelTemplate
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareResultsBrowser import MPDeclareResultsBrowser


import pickle, os
import pandas as pd
import numpy as np


In [2]:
event_log_name = "small"
log_path = f"D:\\LTNcoder\\.out\\eventlogs\\{event_log_name}-0.3-1.xes"

event_log = D4PyEventLog(case_name="case:concept:name")
event_log.parse_xes_log(log_path)

c:\Users\devas\anaconda3\envs\ltn\lib\site-packages\pm4py\util\dt_parsing\parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/5000 [00:00<?, ?it/s]

In [3]:
# Save the conformance checking results to disk to avoid recalculating
def save_conformance_results(conf_check_res, filename=f'{event_log_name}_5000_conformance_results.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(conf_check_res, f)
    print(f"Conformance checking results saved to {filename}")

# Load the conformance checking results from disk
def load_conformance_results(filename=f'{event_log_name}_5000_conformance_results.pkl'):
    try:
        with open(filename, 'rb') as f:
            conf_check_res = pickle.load(f)
        print(f"Conformance checking results loaded from {filename}")
        return conf_check_res
    except FileNotFoundError:
        print(f"File {filename} not found. Run conformance checking first.")
        return None

In [ ]:
if not os.path.exists(f"{event_log_name}-0.3-1.decl"):
    print(f"File {event_log_name}-0.3-1.decl does not exist, running discovery...")
    discovery = DeclareMiner(log=event_log, consider_vacuity=False, min_support=0.05, itemsets_support=0.05, max_declare_cardinality=1)
    declare_model: DeclareModel = discovery.run()
    print(f"Total constraints discovered: {len(declare_model.serialized_constraints)}")
    model_constraints = declare_model.get_decl_model_constraints()
    declare_model.to_file(f"{event_log_name}-0.3-1.decl")

File small-0.3-1.decl does not exist, running discovery...
Computing discovery ...
Total constraints discovered: 2060


In [5]:
if not os.path.exists(f'{event_log_name}_5000_conformance_results.pkl') and event_log is not None and declare_model is not None:
    print(f"File {event_log_name}_5000_conformance_results.pkl does not exist, running conformance checking...")
    basic_checker = MPDeclareAnalyzer(log=event_log, declare_model=declare_model, consider_vacuity=False)
    conf_check_res: MPDeclareResultsBrowser = basic_checker.run()
    save_conformance_results(conf_check_res)
else:
    print(f"Loading conformance checking results from {event_log_name}_5000_conformance_results.pkl")
    conf_check_res = load_conformance_results()
conf_check_df =  conf_check_res.get_metric(metric="state")


File small_5000_conformance_results.pkl does not exist, running conformance checking...
Conformance checking results saved to small_5000_conformance_results.pkl


In [6]:
activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)
satisfied_df = conf_check_res.get_metric(metric="state").fillna(0)

# Support = how often constraint is satisfied
support = satisfied_df.sum(axis=0) / len(satisfied_df)

# Activation rate = how often constraint is activated
activated_counts = activated_df.sum(axis=0)
activation_rate = activated_counts / len(activated_df)

# Satisfied counts
satisfied_counts = satisfied_df.sum(axis=0)

# Confidence = P(satisfied | activated), safe division
confidence = np.where(
    activated_counts != 0,
    satisfied_counts / activated_counts,
    0
)
confidence = pd.Series(confidence, index=activated_counts.index)

# Combine into a DataFrame
metrics_df = pd.DataFrame({
    'support': support,
    'confidence': confidence,
    # 'activation_rate': activation_rate
})
# metrics_df


C:\Users\devas\AppData\Local\Temp\ipykernel_94252\1810640346.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)


In [7]:
# filtered_metrics_df is metrics_df but with rows with support less than 0.2 and in descending order of confidence and no "not" in the constraint name
filtered_metrics_df = metrics_df[~metrics_df.index.str.contains("Not") & (metrics_df['support'] <= 0.2) & (metrics_df['confidence'] >= 0.7)].sort_values(by='confidence', ascending=False)
# filtered_metrics_df = metrics_df[metrics_df['support'] <= 0.2].sort_values(by='confidence', ascending=False)
print("Filtered Metrics DataFrame:")
display(filtered_metrics_df)

Filtered Metrics DataFrame:


,support,confidence
"Responded Existence[Activity M, Activity F] | |",0.0924,0.991416
"Responded Existence[Activity M, Activity K] | |",0.0924,0.991416
"Response[Activity M, Activity F] | |",0.0922,0.989270
"Responded Existence[Activity N, Activity F] | |",0.0928,0.987234
"Responded Existence[Activity M, Activity N] | |",0.0920,0.987124
...,...,...
"Chain Precedence[Activity O, Activity P] | |",0.1558,0.922986
"Chain Response[Activity O, Activity P] | |",0.1556,0.922894
"Chain Precedence[Activity G, Activity H] | |",0.1484,0.922886
"Chain Response[Activity I, Activity J] | |",0.1478,0.921446


In [ ]:
# raise

RuntimeError: No active exception to reraise

# Constraints with low support and high confidence
1. Response[Activity M, Activity F] | |	0.0922	0.9892703862660944
2. Responded Existence[Activity M, Activity F] | |
3. Responded Existence[Activity M, Activity K] | |	0.0924	0.9914163090128756
4. Response[Activity N, Activity F] | |	0.0926	0.9851063829787234

In [9]:
interesting_constraints = [
    # "Response[Approve PO 2, Release PO] | |",
    "Responded Existence[Activity M, Activity F] | |"
    ]

In [10]:
state_df = conf_check_res.get_metric("state")[interesting_constraints]
non_zero_counts = state_df.ne(0).sum(axis=0)
print(non_zero_counts)
non_zero_rows = state_df.index[state_df[interesting_constraints[0]] != 0].tolist()
print(non_zero_rows)
print(len(non_zero_rows))
with open(f'{event_log_name}_ltn_rows.pkl', 'wb') as f:
    pickle.dump(non_zero_rows, f)

Responded Existence[Activity M, Activity F] | |    462
dtype: int64
[3, 7, 34, 43, 55, 73, 77, 83, 103, 112, 113, 114, 121, 129, 132, 151, 152, 160, 169, 171, 177, 180, 191, 207, 236, 238, 255, 261, 262, 269, 325, 326, 391, 409, 413, 452, 458, 464, 477, 490, 496, 504, 533, 534, 543, 548, 550, 556, 571, 572, 583, 623, 626, 636, 638, 641, 648, 649, 667, 684, 696, 701, 740, 748, 750, 757, 763, 764, 779, 791, 794, 816, 818, 826, 827, 838, 843, 844, 850, 858, 860, 864, 866, 875, 909, 912, 915, 918, 921, 922, 948, 950, 961, 962, 968, 973, 975, 978, 1003, 1007, 1022, 1027, 1028, 1048, 1073, 1082, 1106, 1110, 1112, 1113, 1118, 1123, 1144, 1173, 1195, 1197, 1203, 1204, 1206, 1227, 1229, 1245, 1257, 1275, 1283, 1297, 1309, 1330, 1338, 1363, 1366, 1377, 1380, 1406, 1431, 1460, 1461, 1464, 1473, 1494, 1508, 1510, 1511, 1514, 1520, 1550, 1580, 1582, 1611, 1623, 1630, 1635, 1639, 1642, 1643, 1655, 1658, 1659, 1665, 1669, 1682, 1691, 1732, 1734, 1740, 1746, 1756, 1768, 1770, 1814, 1837, 1855, 1868, 1

In [11]:
print("END")

END


In [12]:
# summary_df = conf_check_df.apply(lambda col: col.value_counts()).fillna(0).astype(int)
# summary_df = summary_df.reindex([0, 1])
# summary_df = summary_df / len(conf_check_df)
# summary_df = summary_df.T
# summary_df = summary_df.sort_values(by=1, ascending=False)
# display(summary_df)